In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#Create Emotion Dictionary
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack

# 1. Define File Paths (Based on your Kaggle Screenshot)
# Note: Kaggle input paths usually start with /kaggle/input/

path_joy = "/kaggle/input/normalised-training-data/normalized_td_joy.txt"
path_sadness = "/kaggle/input/normalised-training-data/normalized_td_sadness.txt"
path_sarcasm = "/kaggle/input/normalised-training-data/normalized_td_sarcasm.txt"
path_anger = "/kaggle/input/normalised-training-data/training_data_anger.txt"
path_neutral = "/kaggle/input/normalised-training-data/training_data_neutral.txt"

path_vocab = "/kaggle/input/spelling-x-vocab/Emotion_Vocab.csv.txt"

print("Paths defined successfully.")



In [ ]:
import pandas as pd

# Function to load a file line-by-line (Preserves Emojis & ignores Commas)
def load_data(file_path, label_code):
    try:
        # 1. Open the file using standard Python (handles emojis correctly with utf-8)
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        # 2. Clean the lines:
        # - strip(): removes the invisible \n at the end and spaces
        # - if line.strip(): removes empty lines
        cleaned_lines = [line.strip() for line in lines if line.strip()]
        
        # 3. Convert list to DataFrame
        df = pd.DataFrame({'text': cleaned_lines})
        df['label'] = label_code
        
        return df
        
    except UnicodeDecodeError:
        # Fallback: Sometimes files are not UTF-8. Try 'latin-1' if utf-8 fails.
        try:
            with open(file_path, 'r', encoding='latin-1') as f:
                lines = f.readlines()
            cleaned_lines = [line.strip() for line in lines if line.strip()]
            df = pd.DataFrame({'text': cleaned_lines})
            df['label'] = label_code
            return df
        except Exception as e:
            print(f"Encoding Error on {file_path}: {e}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"General Error loading {file_path}: {e}")
        return pd.DataFrame()

# ---------------------------------------------------------
# RELOAD DATA (Copy this part too)
# ---------------------------------------------------------

# Mapping: Joy=0, Sadness=1, Sarcasm=2, Anger=3, Neutral=4
df_joy = load_data(path_joy, 0)
df_sadness = load_data(path_sadness, 1)
df_sarcasm = load_data(path_sarcasm, 2)
df_anger = load_data(path_anger, 3)
df_neutral = load_data(path_neutral, 4)

# Combine
df_final = pd.concat([df_joy, df_sadness, df_sarcasm, df_anger, df_neutral], axis=0)

# Shuffle
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Cleaned Data Loaded! Total sentences: {len(df_final)}")

# Only run this check IF data was actually loaded
if len(df_final) > 0:
    print(df_final.head())
    print("Shortest sentence length:", df_final['text'].str.len().min())
else:
    print("CRITICAL ERROR: No data was loaded. Check file paths.")

In [ ]:
# 4. Load and Organize the Dictionary with Emojis
try:
    vocab_df = pd.read_csv(path_vocab)
    
    # Detect column names
    col_word = vocab_df.columns[0]      # words
    col_emotion = vocab_df.columns[1]   # emotion labels
    
    # Create the base dictionary
    emotion_dict = {
        "joy": vocab_df[vocab_df[col_emotion] == 'joy'][col_word].tolist(),
        "sadness": vocab_df[vocab_df[col_emotion] == 'sadness'][col_word].tolist(),
        "sarcasm": vocab_df[vocab_df[col_emotion] == 'sarcasm'][col_word].tolist(),
        "anger": vocab_df[vocab_df[col_emotion] == 'anger'][col_word].tolist(),
        "neutral": vocab_df[vocab_df[col_emotion] == 'neutral'][col_word].tolist(),
    }

    # --- ADD RELEVANT EMOJIS ---
    emotion_dict["joy"].extend(["😊","😁","😄","🤩","🥳","😎","💖","✨"])
    emotion_dict["sadness"].extend(["😢","😞","😔","😿","💔","🥺"])
    emotion_dict["anger"].extend(["😡", "😠", "🤬", "👿"])
    emotion_dict["neutral"].extend(["😐","😶","🤔","😑"])
    
    # Sarcasm intentionally left without emojis

    print("Dictionary with emojis created successfully!")
    print(f"Joy words + emojis count: {len(emotion_dict['joy'])}")
    print(f"Sadness words + emojis count: {len(emotion_dict['sadness'])}")
    print(f"Anger words + emojis count : {len(emotion_dict['anger'])}")
    print(f"Neutral words + emojis count : {len(emotion_dict['neutral'])}")
    print(f"Sarcasm words count :{len(emotion_dict['sarcasm'])}")

except Exception as e:
    print("Error loading vocab. Please check column names.", e)

    # fallback dictionary with emojis
    emotion_dict = {
        "joy": ["happy","😊","😁","😄","🤩","🥳","😎","💖","✨"],
        "sadness": ["sad","😢","😞","😔","😿","💔","🥺"],
        "sarcasm": ["yeah","right","wow","totally","favorite","kamal","shabash"],  
        "anger": ["mad","😡","😠","🤬","👿"],
        "neutral": ["😐","😶","🤔","😑"]
    }


# **Important Insights :**


- Td idf define vectors ,give importance to rare words 
- Adding extra word count for each emotion helps model to see those words that appear more and lose their importnace but still not neglected if they are in our emotion dictionary 
- Logistic regression works well with td-idf so it does not claasify a sentence based on stop words
- English stop words can be done but roman urdu stop words have not been created so mnaullay creating them
- We should not limit vocabulary size to some digit and see how many unqiue words we already have 


In [ ]:
# 5. Create Manual Features (Counting dictionary words)

def count_matches(sentence, words):
    if not isinstance(sentence, str): return 0 # Handle empty rows
    sentence_words = sentence.lower().split()
    count = 0
    for w in words:
        if str(w).lower() in sentence_words:
            count += 1
    return count

print("Creating dictionary features...")
# Create a column for each emotion count
df_final['joy_score'] = df_final['text'].apply(lambda x: count_matches(x, emotion_dict.get('joy', [])))
df_final['sad_score'] = df_final['text'].apply(lambda x: count_matches(x, emotion_dict.get('sadness', [])))
df_final['sarc_score'] = df_final['text'].apply(lambda x: count_matches(x, emotion_dict.get('sarcasm', [])))
df_final['ang_score'] = df_final['text'].apply(lambda x: count_matches(x, emotion_dict.get('anger', [])))

# Convert to Matrix
manual_features = df_final[['joy_score', 'sad_score', 'sarc_score', 'ang_score']].values



In [ ]:
# ==========================================
# RE-RUNNING THE PIPELINE TO FIX "NOT FITTED" ERROR
# ==========================================

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# 1. Define your Roman Urdu Stop Words (as discussed)
roman_urdu_stop_words = [
    "hai", "hain", "han", "ho", "hun", "h", "tha", "thi", "thay", "the",
    "ga", "gi", "ge", "raha", "rahi", "rahay", "rha", "rhi",
    "ka", "ki", "ke", "k", "ko", "ne", "se", "sy", "par", "pe", "pr", "per",
    "mein", "me", "mn", "men", "aur", "or", "ao", "bhi", "bh", "to", "toh",
    "liye", "lye", "main", "men", "mjhy", "mujhe", "hum", "hamein",
    "tum", "tu", "ap", "aap", "wo", "woh", "us", "iss", "is", "jo", "jis",
    "ye", "yeh", "apna", "apni", "apne", "kar", "kr", "karna", "krna", "kya",
    "hona", "ho", "hue", "huway", "gaya", "gye", "gai", "gay", "wala", "walay", "wali",
    "sakta", "sakti", "sakte", "wese", "bus", "bas"
]
final_stop_words = list(ENGLISH_STOP_WORDS) + roman_urdu_stop_words

# 2. Re-Initialize and FIT the TF-IDF Vectorizer
print("Fitting TF-IDF on training data...")
# Note: min_df=2 removes typos appearing only once
tfidf = TfidfVectorizer(max_features=None, min_df=2, stop_words=final_stop_words)

# CRITICAL STEP: .fit_transform() learns the vocab AND converts the text
X_tfidf = tfidf.fit_transform(df_final['text'].astype(str)) 
print(f"TF-IDF Fitted. Vocab Size: {len(tfidf.vocabulary_)}")

# 3. Stack with Manual Features
# Ensure 'manual_features' is calculated from your previous step
# If manual_features variable is lost, uncomment the next 2 lines to re-calculate:
# manual_features = df_final[['joy_score', 'sad_score', 'sarc_score', 'ang_score']].values
X_combined = hstack([X_tfidf, manual_features])







 # **Training model with max iteration 1000**

In [ ]:
# 4. Train the Model
print("Training Logistic Regression...")
model = LogisticRegression(max_iter=1000, solver='lbfgs')
model.fit(X_combined, y)
print("Model Trained!")
# ==========================================
# SAVE THE MODEL (Optional but Recommended)
# ==========================================
import joblib

joblib.dump(tfidf, "/kaggle/working/tfidf_vectorizer.pkl")
print("Vectorizer saved!")
joblib.dump(model, "/kaggle/working/roman_urdu_model.pkl")
print("Model saved!")


# **Deleting all files in output comes handy !!**

In [ ]:
import os

folder = "/kaggle/working"

for f in os.listdir(folder):
    full_path = os.path.join(folder, f)
    if os.path.isfile(full_path):
        os.remove(full_path)
        print("Removed:", full_path)


# **Tesing with production rule :using 100 % training data using 100% testing data**

In [ ]:
import numpy as np
from scipy.sparse import hstack

# ==========================================
# PRODUCTION PREDICTION FUNCTION
# ==========================================
def predict_emotion_production(new_sentence, model, tfidf_vectorizer, emotion_dictionary):
    """
    Takes a raw sentence and returns the predicted emotion and confidence score.
    """
    
    # 1. Transform text using the TRAINED vectorizer
    # (Do NOT fit here, only transform)
    tfidf_features = tfidf_vectorizer.transform([new_sentence])
    
    # 2. Calculate Manual Features (The Dictionary Counts)
    # We must maintain the EXACT SAME ORDER as we did during training:
    # Order: Joy, Sadness, Sarcasm, Anger
    
    # Helper to count words (same as before)
    def quick_count(text, word_list):
        words = str(text).lower().split()
        return sum(1 for w in word_list if w in words)

    manual_features = [
        [
            quick_count(new_sentence, emotion_dictionary.get('joy', [])),
            quick_count(new_sentence, emotion_dictionary.get('sadness', [])),
            quick_count(new_sentence, emotion_dictionary.get('sarcasm', [])),
            quick_count(new_sentence, emotion_dictionary.get('anger', []))
        ]
    ]
    
    # Convert to Numpy Array
    manual_features = np.array(manual_features)
    
    # 3. Stack (Glue) them together
    final_input = hstack([tfidf_features, manual_features])
    
    # 4. Predict
    prediction_index = model.predict(final_input)[0]
    probabilities = model.predict_proba(final_input)[0]
    
    # 5. Map to String
    # Update this map if your label numbers are different!
    label_map = {0: "Joy", 1: "Sadness", 2: "Sarcasm", 3: "Anger", 4: "Neutral"}
    predicted_label = label_map.get(prediction_index, "Unknown")
    
    return predicted_label, probabilities

# ==========================================
# HOW TO USE IT
# ==========================================

# Example 1: Sarcasm Test
sentence_1 = "Wow wonderful service, I waited 3 hours for cold food"
label, probs = predict_emotion_production(sentence_1, model, tfidf, emotion_dict)

print(f"Sentence: {sentence_1}")
print(f"Prediction: {label}")
print(f"Confidence: {probs.max():.2f}") # Highest probability
print("-" * 30)

# Example 2: Joy Test
sentence_2 = "I am so happy that I won the prize"
label, probs = predict_emotion_production(sentence_2, model, tfidf, emotion_dict)

print(f"Sentence: {sentence_2}")
print(f"Prediction: {label}")
print(f"Confidence: {probs.max():.2f}")
print("-" * 30)

# Displaying All Results 

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.sparse import hstack

# ======================================================
# STEP 1: LOAD RESOURCES (Model, TF-IDF, Dictionary)
# ======================================================
print("Step 1: Loading Saved Model and Vectorizer...")

# Load the model and vectorizer you saved in the previous step
try:
    model = joblib.load('/kaggle/working/roman_urdu_model.pkl')
    tfidf = joblib.load('/kaggle/working/tfidf_vectorizer.pkl')
    print("✅ Model and TF-IDF loaded successfully from disk.")
except:
    print("⚠️ Could not load from disk. Assuming 'model' and 'tfidf' are already in memory.")

# REDEFINE DICTIONARY (Crucial: Must match training exactly)
# This ensures we count the features exactly the same way
emotion_dict = {
    "joy": ["happy", "fun", "love", "best", "thrilled", "excited", "mubarak"], 
    "sadness": ["sad", "cry", "miss", "bad", "sorrow", "afsos", "dukh"],
    "sarcasm": ["yeah", "right", "wow", "totally", "favorite", "kamal", "shabash"], 
    "anger": ["hate", "angry", "stupid", "worst", "bakwas", "lanat", "ghatiya", "fraud"]
}

# ======================================================
# STEP 2: LOAD THE 5 TEST FILES (From your Screenshot)
# ======================================================
print("\nStep 2: Loading Test Data...")

def load_test_file(filepath, label):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
        return pd.DataFrame({'text': lines, 'label': label})
    except Exception as e:
        print(f"❌ Error loading {filepath}: {e}")
        return pd.DataFrame()

# Paths based on your Kaggle Screenshot
base_path = "/kaggle/input/final-testing-data-n/"

df_test_joy = load_test_file(base_path + "testing_data_n_joy.txt", 0)
df_test_sad = load_test_file(base_path + "testing_data_n_sadnes.txt", 1)
df_test_sarc = load_test_file(base_path + "testing_data_n_sarcasm.txt", 2)
df_test_ang = load_test_file(base_path + "testing_data_n_angerr.txt", 3)
df_test_neu = load_test_file(base_path + "testing_data_n_neutral.txt", 4)

# Combine into one Test DataFrame
df_test = pd.concat([df_test_joy, df_test_sad, df_test_sarc, df_test_ang, df_test_neu], axis=0)
print(f"✅ Total Test Sentences: {len(df_test)}")

# ======================================================
# STEP 3: PREPARE FEATURES (TF-IDF + Manual)
# ======================================================
print("\nStep 3: Preparing Features...")

# A. TF-IDF (Transform ONLY - Do not fit!)
X_test_tfidf = tfidf.transform(df_test['text'].astype(str))

# B. Manual Features (The 4 Columns: Joy, Sad, Sarc, Ang)
def get_manual_counts(text_series):
    features = []
    for text in text_series:
        words = str(text).lower().split()
        counts = [
            sum(1 for w in emotion_dict['joy'] if w in words),
            sum(1 for w in emotion_dict['sadness'] if w in words),
            sum(1 for w in emotion_dict['sarcasm'] if w in words),
            sum(1 for w in emotion_dict['anger'] if w in words)
        ]
        features.append(counts)
    return np.array(features)

X_test_manual = get_manual_counts(df_test['text'])

# C. Stack Them (The "Hybrid" Input)
X_test_final = hstack([X_test_tfidf, X_test_manual])

# ======================================================
# STEP 4: PREDICT & CALCULATE METRICS
# ======================================================
print("\nStep 4: Running Predictions...")
y_true = df_test['label']
y_pred = model.predict(X_test_final)

# Calculate Accuracy
acc = accuracy_score(y_true, y_pred)
print(f"🏆 Overall Test Accuracy: {acc*100:.2f}%")

# Define Label Names for Reports
label_names = ["Joy", "Sadness", "Sarcasm", "Anger", "Neutral"]

# PRINT TEXT REPORT
print("\n" + "="*60)
print("CLASSIFICATION REPORT (Precision, Recall, F1-Score)")
print("="*60)
print(classification_report(y_true, y_pred, target_names=label_names))




In [ ]:
# ======================================================
# STEP 5: VISUALIZATION (Confusion Matrix) & SAVING
# ======================================================
print("Step 5: Generating Visualizations...")

# Create the Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# Plotting
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, 
            yticklabels=label_names,
            linewidths=1, linecolor='black')

plt.title('Final Confusion Matrix (Test Data)', fontsize=15)
plt.ylabel('Actual / True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)

# --- SAVE THE IMAGE BEFORE SHOWING IT ---
save_path = '/kaggle/working/confusion_matrix.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✅ Graph saved successfully at: {save_path}")

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt

report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True)
f1_scores = [report[label]['f1-score'] for label in label_names]

# Create a new figure explicitly
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(label_names, f1_scores, color=['green', 'blue', 'orange', 'red', 'gray'])
ax.set_title("Model Performance by Emotion (F1-Score)")
ax.set_ylabel("F1 Score (Higher is Better)")
ax.set_ylim(0, 1.0)

# Add values on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.01, f"{height:.2f}", ha='center', fontweight='bold')

# Save the figure **before** showing
save_path = '/kaggle/working/F1-score-barchart.png'
fig.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"✅ Bar chart saved successfully at: {save_path}")

plt.show()


In [ ]:
import time
from sklearn.metrics import accuracy_score, f1_score

# ==========================================
# 1. MEASURE TRAINING TIME
# ==========================================
print("⏱️ Measuring Training Time...")
start_train = time.time()

# Retraining just to measure the exact time
model.fit(X_combined, y) 

end_train = time.time()
training_time = end_train - start_train

print(f"✅ Training Time: {training_time:.4f} seconds")

# ==========================================
# 2. MEASURE INFERENCE SPEED
# ==========================================
print("⏱️ Measuring Inference Speed...")
start_pred = time.time()

# Predict on the whole test set (5,278 sentences)
y_pred = model.predict(X_test_final)

end_pred = time.time()
prediction_time = end_pred - start_pred

# Calculate sentences per second
sentences_per_sec = len(y_true) / prediction_time

print(f"✅ Total Prediction Time: {prediction_time:.4f} seconds")
print(f"⚡ Inference Speed: {sentences_per_sec:.0f} sentences/second")

# ==========================================
# 3. GET ACCURACY & F1
# ==========================================
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print("-" * 30)
print(f"YOUR LOGISTIC REGRESSION STATS:")
print(f"Accuracy:        {acc*100:.2f}%")
print(f"F1-Score:        {f1:.2f}")
print(f"Training Time:   {training_time:.2f} sec")
print(f"Inference Speed: {sentences_per_sec:.0f} sent/sec (Very Fast)")
print("-" * 30)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ==========================================
# A. PIE CHART (Dataset Distribution)
# ==========================================
# This shows how many sentences of each emotion were in your Test Set
plt.figure(figsize=(8, 8))
y_true.value_counts().plot.pie(autopct='%1.1f%%', colors=['#ff9999','#66b3ff','#99ff99','#ffcc99', '#c2c2f0'])
plt.title('Test Dataset Distribution (Truth)', fontsize=15)
plt.ylabel('') # Hides the 'label' text on y-axis
plt.savefig('/kaggle/working/pie_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("Pie Chart Saved!")

# ==========================================
# B. BAR CHART (F1-Score per Class)
# ==========================================
# We manually take the numbers from your Classification Report
# Joy=0.17, Sadness=0.11, Sarcasm=0.57, Anger=0.34, Neutral=0.42
emotions = ["Joy", "Sadness", "Sarcasm", "Anger", "Neutral"]
f1_scores = [0.17, 0.11, 0.57, 0.34, 0.42] # Update these if your numbers changed slightly

plt.figure(figsize=(10, 6))
bars = plt.bar(emotions, f1_scores, color=['gray', 'gray', 'green', 'gray', 'blue'])

# Add numbers on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, round(yval, 2), ha='center', fontweight='bold')

plt.title('Model Performance (F1-Score) by Emotion', fontsize=15)
plt.xlabel('Emotion Class')
plt.ylabel('F1 Score')
plt.ylim(0, 1) # F1 is always between 0 and 1
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig('/kaggle/working/f1_bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("Bar Chart Saved!")